In [1]:
import arcpy

aprx = arcpy.mp.ArcGISProject("CURRENT")

print(aprx.filePath)

D:\project_3\Karlsruhe.aprx


In [2]:
fields = arcpy.ListFields(haltestelle)

for field in fields:
    print(field.name, "→", field.type)

NameError: name 'haltestelle' is not defined

In [3]:
maps = aprx.listMaps()

for m in maps:
    print("MAP:", m.name)
    for layer in m.listLayers():
        print("   ", layer.name)

MAP: Map
    haltestelle
    World Topographic Map
    World Hillshade


In [5]:
haltestelle = None

for m in maps:
    for layer in m.listLayers():
        if layer.name == "haltestelle":
            haltestelle = layer
            break

print(haltestelle.dataSource)

D:\project_3\Karlsruhe.gdb\Haltestellennetz\haltestelle


In [6]:
fields = arcpy.ListFields(haltestelle)

for field in fields:
    print(field.name, "→", field.type)

OBJECTID → OID
geom → Geometry
stop_id → String
stop_name → String
parent_station → String


In [7]:
stop_ids = []

with arcpy.da.SearchCursor(haltestelle, ["stop_id"]) as cursor:
    for row in cursor:
        stop_ids.append(row[0])

print("Bekatlar soni:", len(stop_ids))
print("Birinchi 10 ta ID:")
print(stop_ids[:10])

Bekatlar soni: 5865
Birinchi 10 ta ID:
['de:07334:1714:1:1', 'de:07334:1714:1:2', 'de:07334:1721:1:1', 'de:07334:1721:2:2', 'de:07334:1721:3:31', 'de:07334:1721:3:32', 'de:07334:1723:1:1', 'de:07334:1723:2:2', 'de:07334:1723:3:31', 'de:07334:1731:2:3']


In [14]:
import arcpy
import random
from datetime import datetime, timedelta

In [15]:
gdb = arcpy.Describe(haltestelle).path

print(gdb)

D:\project_3\Karlsruhe.gdb\Haltestellennetz


In [16]:
ausstattung = gdb + r"\Ausstattung"

print(ausstattung)

D:\project_3\Karlsruhe.gdb\Haltestellennetz\Ausstattung


In [20]:
ausstattungstypen = [
    "Fahrkartenautomat",
    "Wetterschutzhaus",
    "Beleuchtung",
    "Blindenleitsystem",
    "Dynamische Fahrgastinformation",
    "Fahrradabstellanlage"
]
zustaende = [
    "Neu",
    "Gut",
    "Mittel",
    "Reparaturbeduerftig",
    "Defekt"
]
hersteller = [
    "Siemens",
    "Scheidt & Bachmann",
    "INIT SE",
    "Hoermann"
]
hersteller = [
    "Siemens",
    "Scheidt & Bachmann",
    "INIT SE",
    "Hoermann"
]
heute = datetime.today()

def zufalls_datum():
    tage = random.randint(30, 3650)
    return heute - timedelta(days=tage)
with arcpy.da.InsertCursor(
    ausstattung,
    [
        "AusstattungID",
        "HaltestelleID",
        "Ausstattungstyp",
        "Einbaudatum",
        "Zustand",
        "Hersteller"
    ]
) as cursor:

    nummer = 1

    for stop_id in stop_ids:

        anzahl = random.randint(1, 4)

        for i in range(anzahl):

            ausstattung_id = f"A{nummer:05d}"

            typ = random.choice(ausstattungstypen)
            zustand = random.choice(zustaende)
            firma = random.choice(hersteller)
            datum = zufalls_datum()

            cursor.insertRow([
                ausstattung_id,
                stop_id,
                typ,
                datum,
                zustand,
                firma
            ])

            nummer += 1

print("Ausstattung yaratildi.")
print("Jami:", nummer - 1)

Ausstattung yaratildi.
Jami: 14612


In [18]:
import os

gdb = os.path.dirname(arcpy.Describe(haltestelle).path)

print("GDB:", gdb)

GDB: D:\project_3\Karlsruhe.gdb


In [19]:
ausstattung = os.path.join(gdb, "Ausstattung")

print("Ausstattung:", ausstattung)
print("Exists:", arcpy.Exists(ausstattung))

Ausstattung: D:\project_3\Karlsruhe.gdb\Ausstattung
Exists: True


In [22]:
wartung = os.path.join(gdb, "Wartung")

print(wartung)
print(arcpy.Exists(wartung))

D:\project_3\Karlsruhe.gdb\Wartung
True


In [23]:
ausstattung_ids = []

with arcpy.da.SearchCursor(
    ausstattung,
    ["AusstattungID"]
) as cursor:

    for row in cursor:
        ausstattung_ids.append(row[0])

print("Ausstattung soni:", len(ausstattung_ids))
print(ausstattung_ids[:10])

Ausstattung soni: 14612
['A00001', 'A00002', 'A00003', 'A00004', 'A00005', 'A00006', 'A00007', 'A00008', 'A00009', 'A00010']


In [24]:
wartungstypen = [
    "Inspektion",
    "Reparatur",
    "Wartung",
    "Austausch",
    "Sicherheitspruefung"
]

techniker = [
    "Thomas Mueller",
    "Michael Schneider",
    "Stefan Weber",
    "Andreas Fischer",
    "Daniel Klein"
]
with arcpy.da.InsertCursor(
    wartung,
    [
        "WartungID",
        "AusstattungID",
        "Wartungsdatum",
        "Wartungstyp",
        "Techniker",
        "Kosten",
        "Naechste_Wartung"
    ]
) as cursor:

    nummer = 1

    for ausstattung_id in ausstattung_ids:

        anzahl = random.randint(0, 3)

        for i in range(anzahl):

            wartung_id = f"W{nummer:05d}"

            wartungsdatum = zufalls_datum()

            naechste_wartung = wartungsdatum + timedelta(
                days=random.randint(180, 730)
            )

            typ = random.choice(wartungstypen)
            techniker_name = random.choice(techniker)

            kosten = round(
                random.uniform(50, 2500),
                2
            )

            cursor.insertRow([
                wartung_id,
                ausstattung_id,
                wartungsdatum,
                typ,
                techniker_name,
                kosten,
                naechste_wartung
            ])

            nummer += 1

print("Wartung erstellt.")
print("Jami Wartung:", nummer - 1)

Wartung erstellt.
Jami Wartung: 22006


In [25]:
vertragspartner = os.path.join(gdb, "Vertragspartner")

print(vertragspartner)
print(arcpy.Exists(vertragspartner))

D:\project_3\Karlsruhe.gdb\Vertragspartner
True


In [26]:
vertragspartner_daten = [
    ["VP001", "Siemens Mobility", "Thomas Mueller", "+49 721 100001", "Fahrkartenautomaten"],
    ["VP002", "INIT SE", "Michael Schneider", "+49 721 100002", "Fahrgastinformation"],
    ["VP003", "Hoermann", "Stefan Weber", "+49 721 100003", "Wetterschutz"],
    ["VP004", "Scheidt & Bachmann", "Andreas Fischer", "+49 721 100004", "Fahrkartenautomaten"],
    ["VP005", "Signify", "Daniel Klein", "+49 721 100005", "Beleuchtung"],
    ["VP006", "RTB GmbH", "Thomas Bauer", "+49 721 100006", "Blindenleitsystem"],
    ["VP007", "Strabag", "Markus Wagner", "+49 721 100007", "Haltestelleninfrastruktur"],
    ["VP008", "Kienzler", "Peter Hoffmann", "+49 721 100008", "Fahrradabstellanlagen"],
    ["VP009", "Vossloh", "Frank Richter", "+49 721 100009", "Verkehrsinfrastruktur"],
    ["VP010", "SWARCO", "Christian Wolf", "+49 721 100010", "Verkehrstechnik"]
]
with arcpy.da.InsertCursor(
    vertragspartner,
    [
        "VertragspartnerID",
        "Firmenname",
        "Ansprechpartner",
        "Telefon",
        "Zustaendigkeitsbereich"
    ]
) as cursor:

    for row in vertragspartner_daten:
        cursor.insertRow(row)

print("Vertragspartner erstellt.")
print("Anzahl:", arcpy.management.GetCount(vertragspartner))

Vertragspartner erstellt.
Anzahl: 10


In [27]:
wartungsvertrag = os.path.join(gdb, "Wartungsvertrag")

print(wartungsvertrag)
print(arcpy.Exists(wartungsvertrag))

D:\project_3\Karlsruhe.gdb\Wartungsvertrag
True


In [28]:
def zufalls_vertragsbeginn():
    return heute - timedelta(days=random.randint(30, 5 * 365))
def zufalls_vertragsende(beginn):
    return beginn + timedelta(
        days=random.randint(365, 3 * 365)
    )
vertragsarten = [
    "Wartungsvertrag",
    "Servicevertrag",
    "Rahmenvertrag",
    "Vollwartungsvertrag"
]
vertragspartner_ids = [
    row[0]
    for row in vertragspartner_daten
]

print(vertragspartner_ids)
with arcpy.da.InsertCursor(
    wartungsvertrag,
    [
        "VertragID",
        "AusstattungID",
        "VertragspartnerID",
        "Vertragsbeginn",
        "Vertragsende",
        "Vertragsart",
        "Jaehrliche_Kosten"
    ]
) as cursor:

    nummer = 1

    for ausstattung_id in ausstattung_ids:

        # Har bir uskunaga 1 yoki 2 ta shartnoma
        anzahl = random.randint(1, 2)

        for i in range(anzahl):

            vertrag_id = f"V{nummer:05d}"

            vertragspartner_id = random.choice(
                vertragspartner_ids
            )

            beginn = zufalls_vertragsbeginn()
            ende = zufalls_vertragsende(beginn)

            vertragsart = random.choice(
                vertragsarten
            )

            jaehrliche_kosten = round(
                random.uniform(500, 10000),
                2
            )

            cursor.insertRow([
                vertrag_id,
                ausstattung_id,
                vertragspartner_id,
                beginn,
                ende,
                vertragsart,
                jaehrliche_kosten
            ])

            nummer += 1

print("Wartungsvertrag erstellt.")
print("Anzahl:", nummer - 1)

['VP001', 'VP002', 'VP003', 'VP004', 'VP005', 'VP006', 'VP007', 'VP008', 'VP009', 'VP010']
Wartungsvertrag erstellt.
Anzahl: 21923


In [29]:
print(
    "Ausstattung:",
    arcpy.management.GetCount(ausstattung)
)

print(
    "Vertragspartner:",
    arcpy.management.GetCount(vertragspartner)
)

print(
    "Wartung:",
    arcpy.management.GetCount(wartung)
)

print(
    "Wartungsvertrag:",
    arcpy.management.GetCount(wartungsvertrag)
)

Ausstattung: 14612
Vertragspartner: 10
Wartung: 22006
Wartungsvertrag: 21923
